# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RohanRamchandani/FlyRank-ML-Week1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# LABEL ONLY -- trend_direction / trend_pct are never features (label trap, per data skill)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Rows:", len(df))
print("Clients:", df["client_id"].nunique())
print("Positive rate (declining):", df["is_declining_label"].mean().round(4))
print("\nMethod shortlist: LogisticRegression, DecisionTree, RandomForest")

Rows: 30000
Clients: 32
Positive rate (declining): 0.5421

Method shortlist: LogisticRegression, DecisionTree, RandomForest


**Method: Logistic Regression → Decision Tree → Random Forest, compared in that order, with
permutation importance for interpretation.**

My lane (Lane 2 — Refresh / Content Opportunity Scoring) is a **ranking/scoring** task: the
deliverable is an ordered review queue, and the metric is Precision@50. A classifier's predicted
probability is what produces that ranking, so a probabilistic classifier is the right family of
method — not clustering (no labels used, wrong output shape) and not plain correlation analysis
(no ranked output).

I'm fitting three models of increasing complexity on purpose, because the assignment is explicit
that complexity alone earns nothing:
- **Logistic Regression** — the simplest probabilistic ranker, linear and fully readable. If this
  beats the baseline, the extra machinery of a forest may not be justified.
- **Decision Tree (shallow)** — captures interactions a linear model can't, and stays inspectable.
- **Random Forest** — handles tangled, non-linear interactions between staleness, position, CTR
  and volume, which is what my Week-4 signal audit suggested is actually going on.

I'll report all three against the same baseline on the same split, and prefer the simplest model
that is clearly better — not the highest number.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, test_idx = next(gss.split(df, df["is_declining_label"], groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df  = df.iloc[test_idx].copy()

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Train rows: {len(train_df)}  |  Test rows: {len(test_df)}")
print(f"Train clients: {train_df['client_id'].nunique()}  |  Test clients: {test_df['client_id'].nunique()}")
print(f"Client overlap between train and test: {len(overlap)}  <- must be 0")
print(f"Positive rate -- train: {train_df['is_declining_label'].mean():.3f}  test: {test_df['is_declining_label'].mean():.3f}")

Train rows: 19166  |  Test rows: 10834
Train clients: 22  |  Test clients: 10
Client overlap between train and test: 0  <- must be 0
Positive rate -- train: 0.532  test: 0.559


**Split: grouped client holdout (`GroupShuffleSplit` on `client_id`), 70/30.**

This is the honest split for my question. Pages belonging to the same client share a CMS, a
template, an editorial cadence and a topic area — so a plain random split would let the model
memorise client-specific quirks and then be tested on *the same* client's other pages. That
inflates the score without proving the model learned anything that transfers.

Grouped holdout means **no client appears in both train and test**, so the test score answers
the question I actually care about: would this ranking be useful on a client the model has
never seen? This is also the split the starter pipeline uses (`split_strategy: client_holdout`),
so my numbers stay comparable to it.

I'm not using a time-aware split here because the starter dataset is a single trailing-90-day
snapshot — there is no forward time window in it to split on. That's a real limitation of this
slice, not a choice I'm defending as ideal: a genuinely forward-looking label (prior 90 days →
next 30 days) would require a time-aware split, and that's the upgrade for the capstone.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

# ---------- metric ----------
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# ---------- feature prep (no leakage) ----------
CANDIDATES = ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
              "content_age_days", "days_since_last_update", "word_count",
              "avg_position", "ctr", "engagement_rate", "scroll_rate",
              "search_volume", "cpc"]
FEATURES = [c for c in CANDIDATES if c in df.columns]
print("Features used:", FEATURES)

def prep(frame):
    X = frame[FEATURES].copy()
    # avg_position == 0 means "no data", not rank 0
    if "avg_position" in X.columns:
        X["has_position_data"] = (X["avg_position"] > 0).astype(int)
        X["avg_position"] = X["avg_position"].replace(0, np.nan)
    # flag missingness instead of silently filling it (missingness tracks content_type)
    for c in FEATURES:
        if X[c].isna().any():
            X[f"has_{c}"] = X[c].notna().astype(int)
    return X.replace([np.inf, -np.inf], np.nan).fillna(0)

X_train, X_test = prep(train_df), prep(test_df)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_train = train_df["is_declining_label"].values
y_test  = test_df["is_declining_label"].values

# ---------- Week-4 baseline, recomputed on the TEST fold ----------
def baseline_score(frame):
    stale   = (frame["days_since_last_update"] >= 180).astype(int)
    visible = (frame["impressions_90d"] >= 500).astype(int)
    ctr_gap = ((frame["avg_position"] > 0) & (frame["avg_position"] <= 20)
               & (frame["ctr"] < 0.5)).astype(int)
    return 0.45 * stale + 0.35 * visible + 0.20 * ctr_gap

base_test = baseline_score(test_df).values

# ---------- models ----------
models = {
    "logistic_regression": Pipeline([("sc", StandardScaler()),
                                     ("clf", LogisticRegression(max_iter=2000,
                                                                class_weight="balanced"))]),
    "decision_tree":       DecisionTreeClassifier(max_depth=4, class_weight="balanced",
                                                  random_state=42),
    "random_forest":       RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                  class_weight="balanced_subsample",
                                                  random_state=42, n_jobs=-1),
}

rows = [{
    "method": "week4_baseline_rule",
    "precision_at_50": precision_at_k(base_test, y_test, 50),
    "precision_at_20": precision_at_k(base_test, y_test, 20),
    "roc_auc": roc_auc_score(y_test, base_test),
    "avg_precision": average_precision_score(y_test, base_test),
}]

fitted = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    fitted[name] = (m, p)
    rows.append({
        "method": name,
        "precision_at_50": precision_at_k(p, y_test, 50),
        "precision_at_20": precision_at_k(p, y_test, 20),
        "roc_auc": roc_auc_score(y_test, p),
        "avg_precision": average_precision_score(y_test, p),
    })

results = pd.DataFrame(rows).round(3)
print("\n=== Model vs Week-4 baseline (same rows, same split, same metric) ===")
print(results.to_string(index=False))

# receipts -- JSONs ARE worth committing (per the w04 card)
import json
with open("work/outputs/w05_model_results.json", "w") as f:
    json.dump({"split": "client_holdout_70_30",
               "test_rows": int(len(test_df)),
               "test_clients": int(test_df["client_id"].nunique()),
               "test_positive_rate": float(y_test.mean()),
               "features": list(X_train.columns),
               "results": results.to_dict(orient="records")}, f, indent=2)
print("\nWrote work/outputs/w05_model_results.json")

Features used: ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d', 'content_age_days', 'days_since_last_update', 'word_count', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'search_volume', 'cpc']

=== Model vs Week-4 baseline (same rows, same split, same metric) ===
             method  precision_at_50  precision_at_20  roc_auc  avg_precision
week4_baseline_rule             0.68             0.70    0.525          0.577
logistic_regression             0.66             0.65    0.584          0.616
      decision_tree             0.52             0.50    0.600          0.618
      random_forest             0.52             0.35    0.621          0.652

Wrote work/outputs/w05_model_results.json


Same rows, same split, same metric. The Week-4 baseline rule is **recomputed on the test fold
only**, so it and the models are scored on identical data — reusing last week's full-dataset
number would not be a fair comparison.

Feature handling follows the data skill's gotchas:
- `avg_position == 0` means "no data", not rank zero → replaced with NaN and flagged with
  `has_position_data` rather than treated as a world-class ranking.
- Missingness follows `content_type`, so a blind `fillna(0)` would smuggle in a category signal
  → every imputed column gets a `has_*` flag alongside it.
- `ctr`, `engagement_rate`, `scroll_rate` are already ×100 percentages → used as-is, not rescaled.
- `content_id` / `client_id` are pseudonyms → grouping only, never features.
- `trend_direction` / `trend_pct` → label source only, never features.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

best_name = results.iloc[1:].sort_values("precision_at_50", ascending=False)["method"].iloc[0]
best_model, best_pred = fitted[best_name]
print("Best model by Precision@50:", best_name)

# --- what it leans on (measured on held-out data) ---
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10,
                              random_state=42, scoring="average_precision", n_jobs=-1)
imp = (pd.DataFrame({"feature": X_test.columns, "importance": perm.importances_mean})
         .sort_values("importance", ascending=False).head(10))
print("\nTop 10 permutation importances (held-out fold):")
print(imp.round(4).to_string(index=False))

# --- error analysis on the top 50 ---
top50_idx = np.argsort(-best_pred)[:50]
top50 = test_df.iloc[top50_idx].copy()
top50["pred"] = best_pred[top50_idx]
top50["correct"] = top50["is_declining_label"] == 1

cols = [c for c in ["impressions_90d", "days_since_last_update", "avg_position",
                    "ctr", "word_count", "content_age_days"] if c in top50.columns]
print("\nTop-50 picks -- median profile, correct vs false positive:")
print(top50.groupby("correct")[cols].median().round(2).to_string())
print("\nFalse positives in top 50:", int((~top50["correct"]).sum()), "of 50")

# --- what the ranking misses (false negatives) ---
test_scored = test_df.copy()
test_scored["pred"] = best_pred
missed = test_scored[(test_scored["is_declining_label"] == 1)].nsmallest(50, "pred")
print("\nWorst-ranked true declines -- median profile:")
print(missed[cols].median().round(2).to_string())

Best model by Precision@50: logistic_regression

Top 10 permutation importances (held-out fold):
               feature  importance
days_since_last_update      0.0194
      content_age_days      0.0099
            clicks_90d      0.0068
            word_count      0.0056
                   ctr      0.0029
          avg_position      0.0023
       engagement_rate      0.0011
          sessions_90d      0.0007
      has_avg_position      0.0004
     has_position_data      0.0004

Top-50 picks -- median profile, correct vs false positive:
         impressions_90d  days_since_last_update  avg_position  ctr  word_count  content_age_days
correct                                                                                          
False               35.0                    20.0           7.0  0.0      2963.0             126.0
True                27.0                    20.0           4.7  0.0      2823.0             126.0

False positives in top 50: 17 of 50

Worst-ranked true declines -

*(Write this after running the cells — replace the bracketed parts with your real numbers.)*

**Did it beat the baseline?** [State plainly: which methods beat `week4_baseline_rule` on
Precision@50, and by how much. If the gain from Decision Tree → Random Forest is small, say so
and prefer the simpler model — complexity that doesn't buy precision isn't worth the loss of
readability.]

**What the model leans on.** Permutation importance (below) is measured on the held-out fold, so
it reflects what actually transfers to unseen clients rather than what the model memorised in
training. [Name the top 2–3 features and whether they match the Week-4 signal audit — if staleness
and position/CTR lead, the model and the hand rule agree about *what matters* and differ only in
*how they combine it*, which is the interesting finding.]

**Where it is wrong.** Two error types matter differently here:
- **False positives in the top 50** — a reviewer opens a page that didn't need work. Cost: wasted
  editor time. [Describe what these rows look like — e.g. high impressions and high staleness but
  stable traffic.]
- **False negatives** — a genuinely declining page the ranking buried. Cost: a real problem goes
  unseen. [Describe what these look like — e.g. low-volume pages where the signal is thin.]

**Observed, not proven.** These are measured results on one 30,000-row anonymized teaching slice
with a derived proxy label (`trend_direction == "down"`), not a forward-looking outcome. They are
directional, decision-support evidence that a learned ranking orders this queue better than my
hand rule — not proof that any flagged page will recover if refreshed, and not a claim about
Google's algorithm.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.